# Ders 07 - Planlama Tasarım Deseni

Bu not defteri, Microsoft Agent Framework kullanarak AI ajanları için **Planlama Tasarım Deseni**ni göstermektedir.
Karmaşık seyahat taleplerini yapılandırılmış alt görevlere nasıl böleceğinizi, bunları uzman ajanlara nasıl atayacağınızı
ve ortaya çıkan planı nasıl yürüteceğinizi öğreneceksiniz — hepsi Pydantic modelleriyle desteklenen yapılandırılmış çıktı kullanılarak.


## Kurulum


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Görev Parçalama

Görev parçalama, planlama tasarım deseninin temelidir. Tek bir ajandan karmaşık bir isteği baştan sona ele almasını istemek yerine, problemi daha küçük, iyi tanımlanmış **alt görevlere** böleriz. Her alt görev, net önceliklere ve bağımlılık sırasına sahip uzman bir ajana (ör. uçuşlar, oteller, aktiviteler) atanır.

Bu yaklaşım birkaç fayda sağlar:
- **Açıklık**: her alt görevin tek bir sorumluluğu vardır.
- **Paralellik**: bağımsız alt görevler eşzamanlı olarak çalışabilir.
- **Güvenilirlik**: hatalar bireysel alt görevlerle sınırlıdır.
- **Bütçe takibi**: maliyetler alt görev bazında tahmin edilir ve toplanır.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Yapılandırılmış Çıktı ile Planlama Ajanı Oluşturma

Planlama ajanı, bir **resepsiyon koordinatörü** olarak görev yapar. Yüksek seviyeli bir seyahat talebi verildiğinde,
talebi alt görevlere ayırarak, öncelikleri belirleyerek ve bir konsiyerj veya yürütme katmanının işi gerçekleştirebilmesi için bağımlılıkları tanımlayarak yapılandırılmış bir `TravelPlan` oluşturur.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Uzman Araçlarla Bir Planı Yürütme

Resepsiyon görevlisi yapılandırılmış bir plan oluşturduktan sonra, **konserje ajanı** bunu uygular. Her uzman araç, bir alt görev kategorisini (uçuşlar, oteller, aktiviteler) ele alır. Konserje, planın alt görevlerini bağımlılık sırasına göre yineleyerek her birini uygun araca gönderir.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Özeti

Bu derste AI ajanları için **Planlama Tasarım Deseni** öğrendiniz:

1. **Görev Parçalama** — Bir resepsiyon planlama ajanı, karmaşık bir seyahat isteğini Pydantic modelleri kullanarak yapılandırılmış alt görevlere bölerek her birini öncelikler ve bağımlılıklar ile uzman bir ajana atar.
2. **Yapılandırılmış Çıktı** — `response_format` geçilerek ajan, serbest biçimli metin yerine doğrulanmış bir `TravelPlan` nesnesi döndürür ve böylece sonraki işlemleri güvenilir hale getirir.
3. **Planın Yürütülmesi** — Bir konsiyerj ajan, planı uygulamak ve sonuçları raporlamak için uzman araçlar (`book_flight`, `reserve_hotel`, `book_activity`) kullanarak alt görevlerin üzerinden geçer.

Bu desen, *ne yapılacağını* (planlama) ve *nasıl yapılacağını* (yürütme) ayırarak ajanları daha modüler, test edilebilir ve kolay genişletilebilir hale getirir.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:  
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba göstermekle birlikte, otomatik çevirilerde hatalar veya yanlışlıklar olabileceğini lütfen unutmayınız. Orijinal belge, kendi ana dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi tavsiye edilir. Bu çevirinin kullanımıyla ortaya çıkabilecek yanlış anlamalar veya yorum farklılıklarından dolayı sorumluluk kabul edilmemektedir.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
